<img src="../docs/banniere_projets_GM.png" width="70%" /><br>

-----
<img src="../docs/kayak_logo.png" width="30%" /><br>

# Plan your trip with Kayak

>Ce projet est réalisé dans le cadre du bloc 1 (Construction et alimentation d'une infrastructure de gestion de données) de la certification CDSD | Concepteur développeur en science des données | RNCP35288 | JEDHA

---
## 🚧 Projet

L'équipe marketing de Kayak a besoin d'aide pour un nouveau projet. Après avoir mené une étude auprès des utilisateurs, elle a constaté que 70 % d'entre eux, qui prévoient un voyage, souhaiteraient obtenir davantage d'informations sur leur destination.

De plus, les études auprès des utilisateurs montrent que les gens ont tendance à être sceptiques quant aux informations qu'ils lisent s'ils ne connaissent pas la marque qui a produit le contenu.

L'équipe marketing souhaite donc créer une application qui recommandera aux utilisateurs des destinations pour leurs prochaines vacances.<br>
Cette application devra s'appuyer sur des données réelles concernant :

* La météo
* Les hôtels dans la région

L'application devrait alors être en mesure de recommander les meilleures destinations et les meilleurs hôtels en fonction des variables ci-dessus à tout moment.
<br><br>

## 🎯 Objectifs

Le projet venant de démarrer, votre équipe ne dispose d'aucune donnée permettant de créer cette application.<br>
Votre tâche consistera donc à :

* Récupérer des données à partir de destinations (35 villes françaises pré-sélectionnées) via l'API de https://nominatim.org
* Obtenez des données météorologiques pour chaque destination via l'API de https://openweathermap.org
* Obtenez des informations sur les hôtels pour chaque destination via un scraping de https://booking.com
* Stockez toutes les informations ci-dessus sous forme de fichier csv dans un data lake sous AWS S3
* Extrayez, transformez et chargez (ETL) les données nettoyées de votre data lake vers un entrepôt de données (data warehouse) sous NeonDB.


---
Ce notebook orchestre la chaîne complète : **collecte → data lake → ETL → data warehouse → visualisation**.

Le code "métier" est rangé dans le dossier `src/` (un module par étape) et ce notebook
se contente de l'appeler dans l'ordre.<br>
C'est volontaire : on garde le notebook lisible et on peut réutiliser/tester les fonctions ailleurs.

---

### Plan de l'infrastructure

| Étape | Outil | Rôle |
|-------|-------|------|
| Collecte météo | Nominatim + OpenWeatherMap | GPS + prévisions 7 jours |
| Collecte hôtels | Selenium + Parsel | top 3 hôtels Booking par ville |
| Data lake | AWS S3 | stockage brut, peu coûteux, scalable |
| ETL | Python (pandas + psycopg2) | extract / transform / load |
| Data warehouse | Neon DB (PostgreSQL serverless) | données propres et requêtables |
| Visualisation | Plotly | cartes des destinations et hôtels |


---

## 1 - Imports de librairies & configuration

On rend le dossier `src/` importable, puis on charge le fichier `.env` qui contient **toutes les clés secrètes** (API météo, identifiants AWS, accès base).<br>
Ce fichier n'est jamais versionné (il est dans le `.gitignore`) : c'est important pour la sécurité et la conformité RGPD.

In [1]:
import sys
from pathlib import Path

# On ajoute la racine du projet au chemin Python pour pouvoir faire `from src import ...`
RACINE = Path.cwd().parent
sys.path.append(str(RACINE))

from dotenv import load_dotenv
load_dotenv(RACINE / ".env")   # charge les variables d'environnement secrètes

from src import config, weather, hotels, data_lake, etl, viz

print(f"{len(config.VILLES)} villes cibles chargées")
print("Modules importés ✓")

35 villes cibles chargées
Modules importés ✓


---

## 2 - Collecte des données météo

Pour chaque ville, le module `weather` :
1. trouve les coordonnées GPS via **Nominatim** (gratuit, sans clé)
2. récupère les prévisions 7 jours via **OpenWeatherMap One Call 4.0** (clé requise)
3. calcule un **beauty score** (0–100) qui résume la qualité du temps

In [2]:
import requests, os

api_key = os.getenv("api_key")
url = "https://api.openweathermap.org/data/2.5/forecast"
params = {"lat": 48.85, "lon": 2.35, "appid": api_key, "units": "metric"}
r = requests.get(url, params=params)
print(r.status_code)
print(r.json())


200
{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1783965600, 'main': {'temp': 33.08, 'feels_like': 31.82, 'temp_min': 28.45, 'temp_max': 33.08, 'pressure': 1012, 'sea_level': 1012, 'grnd_level': 1003, 'humidity': 28, 'temp_kf': 4.63, 'dew_point': 12.14}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10d'}], 'clouds': {'all': 58}, 'wind': {'speed': 2.55, 'deg': 129, 'gust': 5.56}, 'visibility': 10000, 'pop': 1, 'rain': {'3h': 0.57}, 'sys': {'pod': 'd'}, 'dt_txt': '2026-07-13 18:00:00'}, {'dt': 1783976400, 'main': {'temp': 30.82, 'feels_like': 29.74, 'temp_min': 26.3, 'temp_max': 30.82, 'pressure': 1012, 'sea_level': 1012, 'grnd_level': 1004, 'humidity': 32, 'temp_kf': 4.52, 'dew_point': 12.23}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04n'}], 'clouds': {'all': 72}, 'wind': {'speed': 2.41, 'deg': 141, 'gust': 4.65}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2026-07-13 21:00:00'}, {'d

In [3]:
import os

api_key = os.getenv("api_key")   # clé OpenWeatherMap

df_weather = weather.collecter_meteo(api_key)
df_weather.head()

  [OK] Mont Saint Michel              weather= 80.8
  [OK] St Malo                        weather= 76.5
  [OK] Bayeux                         weather= 69.2
  [OK] Le Havre                       weather= 73.9
  [OK] Rouen                          weather= 88.4
  [OK] Paris                          weather= 78.9
  [OK] Amiens                         weather= 65.0
  [OK] Lille                          weather= 86.2
  [OK] Strasbourg                     weather= 78.6
  [OK] Chateau du Haut Koenigsbourg   weather= 78.1
  [OK] Colmar                         weather= 76.8
  [OK] Eguisheim                      weather= 76.9
  [OK] Besancon                       weather= 83.7
  [OK] Dijon                          weather= 86.9
  [OK] Annecy                         weather= 78.6
  [OK] Grenoble                       weather= 79.3
  [OK] Lyon                           weather= 69.3
  [OK] Gorges du Verdon               weather= 92.6
  [OK] Bormes les Mimosas             weather= 82.2
  [OK] Cassi

,id,city,latitude,longitude,country,fetch_date,forecast_end_date,avg_temp_7j,avg_humidity_7j,total_rain_7j,avg_pop_7j,weather_score
0,1,Mont Saint Michel,48.635954,-1.511460,France,2026-07-13,2026-07-17,23.61,64.08,11.39,0.27,80.77
1,2,St Malo,48.649518,-2.026041,France,2026-07-13,2026-07-17,21.50,79.03,13.53,0.33,76.47
2,3,Bayeux,49.276462,-0.702474,France,2026-07-13,2026-07-17,20.98,78.00,25.80,0.20,69.15
3,4,Le Havre,49.493898,0.107973,France,2026-07-13,2026-07-17,21.59,72.65,20.40,0.27,73.93
4,5,Rouen,49.440459,1.093966,France,2026-07-13,2026-07-17,25.57,49.12,4.46,0.14,88.41


### Comment est calculé le `weather score` ?

On part de 100 points et on retire des pénalités sur 3 critères, puis on fait la moyenne :

- **température** : on s'éloigne de 20 °C → on perd des points ;
- **humidité** : au-delà de 40 %, on perd des points ;
- **pluie** : chaque mm cumulé sur la semaine coûte des points.

Les réglages (idéaux et pénalités) sont dans `src/config.py`, faciles à ajuster selon
sa propre définition du "beau temps".

In [4]:
# Aperçu trié : les villes au meilleur temps prévu
df_weather.sort_values("weather_score", ascending=False).head(10)

,id,city,latitude,longitude,country,fetch_date,forecast_end_date,avg_temp_7j,avg_humidity_7j,total_rain_7j,avg_pop_7j,weather_score
17,18,Gorges du Verdon,43.749656,6.328562,France,2026-07-13,2026-07-17,25.93,44.33,0.00,0.00,92.63
29,30,Ariege,42.945537,1.406554,France,2026-07-13,2026-07-17,24.58,47.90,0.84,0.04,92.23
4,5,Rouen,49.440459,1.093966,France,2026-07-13,2026-07-17,25.57,49.12,4.46,0.14,88.41
30,31,Toulouse,43.604464,1.444243,France,2026-07-13,2026-07-17,29.57,48.23,1.11,0.07,86.95
13,14,Dijon,47.321581,5.041470,France,2026-07-13,2026-07-17,26.22,48.62,5.95,0.19,86.94
21,22,Aix en Provence,43.529842,5.447474,France,2026-07-13,2026-07-17,30.31,48.95,0.00,0.00,86.71
31,32,Montauban,44.017584,1.354999,France,2026-07-13,2026-07-17,29.39,48.45,2.06,0.15,86.42
7,8,Lille,50.636565,3.063528,France,2026-07-13,2026-07-17,25.28,53.75,5.85,0.19,86.24
23,24,Uzes,44.012128,4.419672,France,2026-07-13,2026-07-17,29.55,54.70,0.16,0.02,85.45
24,25,Nimes,43.837425,4.360069,France,2026-07-13,2026-07-17,29.84,54.50,0.00,0.00,85.32


---

## 3 - Collecte des hôtels (scraping Booking.com)

Booking n'a pas d'API publique : on **scrape** la page de résultats avec Selenium
(le contenu est chargé en JavaScript) puis on parse le HTML avec Parsel.

Pour chaque ville, on récupère le **top 3** des hôtels triés par note utilisateurs :
nom, note, description, lien.

>> <u>**Attention !**</u>
>>
>> - Étape longue (un navigateur par ville).
>>
>>- Les classes CSS de Booking changent régulièrement : si 0 hôtel revient, il faut mettre à jour les sélecteurs dans `src/hotels.py`.
>>
>>- **RGPD** : on ne collecte que des données *publiques* sur des *établissements* (et non sur des personnes). Aucune donnée personnelle d'utilisateur n'est traitée.

In [5]:
df_hotels = hotels.scraper_toutes_villes()
df_hotels.head()

  Scraping : Mont Saint Michel
  Scraping : St Malo
  Scraping : Bayeux
  Scraping : Le Havre
  Scraping : Rouen
  Scraping : Paris
  Scraping : Amiens
  Scraping : Lille
  Scraping : Strasbourg
  Scraping : Chateau du Haut Koenigsbourg
  Scraping : Colmar
  Scraping : Eguisheim
  Scraping : Besancon
  Scraping : Dijon
  Scraping : Annecy
  Scraping : Grenoble
  Scraping : Lyon
  Scraping : Gorges du Verdon
  Scraping : Bormes les Mimosas
  Scraping : Cassis
  Scraping : Marseille
  Scraping : Aix en Provence
  Scraping : Avignon
  Scraping : Uzes
  Scraping : Nimes
  Scraping : Aigues Mortes
  Scraping : Saintes Maries de la mer
  Scraping : Collioure
  Scraping : Carcassonne
  Scraping : Ariege
  Scraping : Toulouse
  Scraping : Montauban
  Scraping : Biarritz
  Scraping : Bayonne
  Scraping : La Rochelle


,id,city,hotel_name,note,description,lien
0,1,Mont Saint Michel,Hôtel Vert,8.2,L’Hotel Vert vous propose des chambres décorée...,https://www.booking.com/hotel/fr/vert.fr.html?...
1,2,Mont Saint Michel,Le Relais Saint Michel,8.3,Le Relais Saint Michel vous accueille face à l...,https://www.booking.com/hotel/fr/le-relais-sai...
2,3,Mont Saint Michel,Mercure Mont Saint Michel,8.4,Installé dans des espaces verts à seulement 2 ...,https://www.booking.com/hotel/fr/mont-saint-mi...
3,4,St Malo,T2 terrasse de charme à Rochebonne 400 m plage,9.6,L’hébergement T2 terrasse de charme à Rochebon...,https://www.booking.com/hotel/fr/t2-de-charme-...
4,5,St Malo,Le Solidor vue mer,9.3,L’hébergement Le Solidor vue mer vous accueill...,https://www.booking.com/hotel/fr/le-solidor-vu...


---

## 4 - Sauvegarde locale puis dépôt dans le data lake (S3)

On sauvegarde d'abord en CSV local (utile pour déboguer et garder une trace), puis on envoie ces fichiers dans le **bucket S3**.<br>
S3 sert de *data lake* : un stockage bon marché, capable d'absorber de gros volumes, où l'on dépose les données avant de les raffiner.

In [6]:
# Sauvegarde locale dans data/processed/
config.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_weather.to_csv(config.WEATHER_CSV, index=False)
df_hotels.to_csv(config.HOTELS_CSV, index=False)
print("CSV sauvegardés localement : OK")

# Dépôt dans le data lake S3
data_lake.uploader_csv(config.WEATHER_CSV, "villes_meteo.csv")
data_lake.uploader_csv(config.HOTELS_CSV, "df_top3_hotels.csv")

CSV sauvegardés localement : OK
  [OK] upload : s3://bucket-geoffrey-dsfs-ft-38/projets-certification-cdsd/projet-kayak_GM/villes_meteo.csv
  [OK] upload : s3://bucket-geoffrey-dsfs-ft-38/projets-certification-cdsd/projet-kayak_GM/df_top3_hotels.csv


---

## 5 - Processus ETL : du data lake vers le data warehouse

Le module `ETL` enchaîne les trois étapes :

1. **Extract** : relit les CSV depuis S3

2. **Transform** : supprime les doublons, vérifie les plages de valeurs
  (weather_score compris entre [0–100], note comprise entre [0–10]), contrôle les coordonnées GPS et nettoie le texte

3. **Load** : (re)crée les tables et insère les données propres dans **Neon DB** (PostgreSQL).

Le pipeline est **rejouable** : on vide les tables avant de recharger, donc le relancer ne crée pas de doublons.

In [7]:
# Pipeline complet en une fonction (Extract → Transform → Load → vérification)
etl.run_etl()

  Extract : 35 villes, 105 hôtels
  Transform : 35 villes, 105 hôtels après nettoyage
  [OK] Tables weather et hotels créées
  [OK] Load : 35 villes, 105 hôtels insérés
  weather : 35 lignes  |  hotels : 105 lignes
  Top 5 destinations :
    Gorges du Verdon               92.6
    Ariege                         92.2
    Rouen                          88.4
    Toulouse                       87.0
    Dijon                          86.9


>> <u>**Observation :**</u> Pourquoi un entrepôt SQL après le data lake ?
>>
>> Le data lake (S3) stocke des fichiers bruts : pratique pour archiver, mais pénible à interroger.<br>
>>
>>Le data warehouse (PostgreSQL) contient les données **nettoyées et structurées**, que les analystes peuvent interroger en SQL simplement.

---

## 6 - Visualisation des résultats

Deux cartes Plotly, conformes au livrable demandé :
- **Carte 1** : les 5 destinations incontournables (meilleur weather score)

- **Carte 2** : les 20 hôtels les mieux notés

Le module `viz` génère les figures **et les sauvegarde** dans `reports/figures/`
(HTML interactif + PNG si `kaleido` est installé).

In [8]:
# On repart des CSV propres (ou on pourrait relire depuis la base)
df_weather = __import__("pandas").read_csv(config.WEATHER_CSV)
df_hotels  = __import__("pandas").read_csv(config.HOTELS_CSV)

# Génère et sauvegarde les 2 cartes dans reports/figures/
viz.generer_toutes_figures(df_weather, df_hotels)

  [OK] C:\Users\geoff\OneDrive\Documents\NOTEBOOKS_Data Science_JEDHA\Data Science FULLSTACK_Nov-25-Fev-26\PROJETS_Blocs\Bloc-1-Construction_et_alimentation_d_une_infrastructure_de_gestion_de_donnees\Projet-CDSD_Bloc-1_Kayak\reports\figures\carte_top5_destinations.html
  [i] PNG non généré (No module named 'kaleido.errors'). Installe 'kaleido' si besoin.
  [OK] C:\Users\geoff\OneDrive\Documents\NOTEBOOKS_Data Science_JEDHA\Data Science FULLSTACK_Nov-25-Fev-26\PROJETS_Blocs\Bloc-1-Construction_et_alimentation_d_une_infrastructure_de_gestion_de_donnees\Projet-CDSD_Bloc-1_Kayak\reports\figures\carte_top20_hotels.html
  [i] PNG non généré (No module named 'kaleido.errors'). Installe 'kaleido' si besoin.


In [10]:
# Affichage de la carte des destinations (villes) en fonction du weather_score
fig1 = viz.carte_top_destinations(df_weather, df_hotels, n=35)
fig1.show()

In [11]:
# Affichage de la carte du top 20 des hôtels
fig2 = viz.carte_top_hotels(df_weather, df_hotels, n=20)
fig2.show()

---

## 7 - Conclusion

Nous avons construit une infrastructure de données complète et simple :

1. **collecte** (Nominatim + OpenWeatherMap + scraping Booking)
2. **data lake** (S3)
3. **ETL** (nettoyage/validation)
4. **data warehouse** (Neon DB)
5. **visualisation** (Plotly)

Les figures finales sont dans `reports/figures/`.<br>
Le code est découpé par responsabilité dans `src/`, ce qui rend chaque étape testable et réutilisable.
<br><br>

---
---